# Symbolic Calculation for *Non-concurrent mutation and unidirectional influence*

## Initialization

### Module

In [5]:
from sympy import *
from sympy.abc import i, j, k, l, r, N, n, m 
import math

### Symbols

In [7]:
# mu = N*u ➡ mutation rate of preference
# nu = N*v ➡ mutation rate of phenotype
u, v = symbols('u, v', real = True, positive = True)
mu, nu = symbols('mu, nu', real = True, positive = True)

In [8]:
# coalesent time
tau_2, tau_3 = symbols('tau_2, tau_3', real = True, positive = True)

In [9]:
# N ➡ population size
# n ➡ the number of strategies
# r_1 ➡ the number of phenotypes in layer-1
# r_2 ➡ the number of phenotypes in layer-2
# K ➡ 2K+1 is the number of associated phenotypes
N, n, r_1, r_2, K = symbols('N, n, r_1, r_2, K', real = True, positive = True)

In [10]:
# strategies
p_1, p_2, q_1, q_2 = symbols('p_1, p_2, q_1, q_2',real = True, positive = True)
p, q= symbols('p, q',real = True, positive = True)

In [11]:
# payoffs
R_1, S_1, T_1, P_1 = symbols('R_1, S_1, T_1, P_1',real = True, positive = True)
R_2, S_2, TT_2, P_2 = symbols('R_2, S_2, T_2, P_2',real = True, positive = True)
#The variable name here is TT_2, which is used to easily distinguish it from the subsequent function name T_2 in the code

In [12]:
# selection strength
beta = symbols('beta',real = True, positive = True)

## Function

### The probability density of coalesent time $\tau_2$
>\begin{equation}
    T_2(\tau_2) = e^{-\tau_2} \notag
\end{equation}

In [15]:
def T_2(tau):
    return exp(-tau)

### The probability density of two steps coalesent time $\tau_2+\tau_3$
> \begin{align}
    T_3(\tau_2, \tau_3) = 3e^{-3\tau_3}e^{-\tau_2} \notag
\end{align}

In [17]:
def T_3(tau_2, tau_3):
    return 3*exp(-3*tau_3)*exp(-tau_2)

### The probability that two people have the same strategy since their coalescent time $\tau$
>\begin{align}
    s_2(\tau) &= e^{-\mu \tau} + \dfrac{1 - e^{-\mu \tau}}{n} \notag
\end{align}

In [19]:
def s_2(tau):
    return exp(-mu*tau) + (1 - exp(-mu*tau))/n

### The probability that three people have the same strategy since their coalescent time $\tau_2 + \tau_3$
> \begin{align}
    s_3(\tau_2,\tau_3) = \dfrac{1}{n^2} \left[ s_2(\tau_2)\left( 1+3(n-1)e^{-\mu \tau_3}+(n-1)(n-2)e^{-\frac{3}{2} \mu \tau_3} \right)+(1-s_2(\tau_2)) \left( 1+(n-3)e^{-\mu \tau_3}-(n-2)e^{-\frac{3}{2} \mu \tau_3} \right) \right] \notag
\end{align}

In [21]:
def s_3(tau_2, tau_3):
    s_2_tau_2 = s_2(tau_2)
    return (1/(n**2))*(s_2_tau_2*(1+3*(n-1)*exp(-mu*tau_3)+(n-1)*(n-2)*exp(-3/2 * mu*tau_3)) + (1-s_2_tau_2)*(1+(n-3)*exp(-mu*tau_3)-(n-2)*exp(-3/2 * mu*tau_3)))


### The probability that two people have the same phenotype since their coalescent time $\tau$ on each layer
>\begin{align}
    g_1(\tau) = e^{-\nu \tau} + \dfrac{1 - e^{-\nu \tau}}{r_1} \qquad\qquad\qquad g_2(\tau) = e^{-\nu \tau} + \dfrac{1 - e^{-\nu \tau}}{2K+1}\notag  
\end{align}

In [23]:
def g_1(tau):
    return exp(-nu*tau) + (1 - exp(-nu*tau))/r_1

# R_K = 2K+1
R_K = symbols('R_K', real = True, positive = True)
def g_2(tau):
    return exp(-nu*tau) + (1 - exp(-nu*tau))/R_K

### The joint probabilities $Q_{11}(\tau)$, $Q_{10}(\tau)$ and $Q_{01}(\tau)$ --- Eq.(S46)

In [25]:
def Q_11(tau):
    return g_1(tau)*g_2(tau)

def Q_10(tau):
    return g_1(tau)*(1-g_2(tau))

def Q_01(tau):
    return (1-g_1(tau))*g_2(tau)

## Calculation for pair and triplet correlations --- Eq.(S29)~(S34)

### Example
> \begin{align}
    \langle x_i^{k_1k_2} x_{\ast}^{k_1 k_2} \rangle &= \dfrac{1}{n} \int_{0}^{+\infty} T_2(\tau_2)Q_{11}(\tau_2) \; d\tau_2 \notag \\
    \notag \\
    \langle x_i^{k_1 k_2} x_i^{k_1 k_2} \rangle &= \dfrac{1}{n} \int_{0}^{+\infty} T_2(\tau_2)s_2(\tau_2)Q_{11}(\tau_2) \; d\tau_2 \notag \\
    \notag \\
    \langle x_i^{k_1k_2} x_j^{k_1 k_2} \rangle &= \dfrac{1}{n-1} \left( \langle x_i^{k_1 k_2} x_{\ast}^{k_1 k_2} \rangle - \langle x_i^{k_1 k_2} x_i^{k_1 k_2} \rangle \right) \notag
\end{align}

In [28]:
X_i_k1k2_X___k1k2 = integrate(T_2(tau_2)*Q_11(tau_2), (tau_2, 0, oo))/n
X_i_k1k2_X___k1k2 = X_i_k1k2_X___k1k2.nsimplify()
X_i_k1k2_X___k1k2 = factor(X_i_k1k2_X___k1k2)

X_i_k1k2_X_i_k1k2 = integrate(T_2(tau_2)*s_2(tau_2)*Q_11(tau_2), (tau_2, 0, oo))/n
X_i_k1k2_X_i_k1k2 = X_i_k1k2_X_i_k1k2.nsimplify()
X_i_k1k2_X_i_k1k2 = factor(X_i_k1k2_X_i_k1k2)

X_i_k1k2_X_j_k1k2 = (1/(n-1))*(X_i_k1k2_X___k1k2 - X_i_k1k2_X_i_k1k2)
X_i_k1k2_X_j_k1k2 = X_i_k1k2_X_j_k1k2.nsimplify()
X_i_k1k2_X_j_k1k2 = factor(X_i_k1k2_X_j_k1k2)

### Others

In [30]:
X_i_X_i_k1k2_X___k1k2_in = T_3(tau_2, tau_3)*(s_2(tau_3)*Q_11(tau_2+tau_3)+s_2(tau_2+tau_3)*Q_11(tau_3)+s_2(tau_2+tau_3)*Q_11(tau_2+tau_3))
X_i_X_i_k1k2_X___k1k2 = integrate(X_i_X_i_k1k2_X___k1k2_in, (tau_2, 0, oo), (tau_3, 0, oo))/(3*n)
X_i_X_i_k1k2_X___k1k2 = X_i_X_i_k1k2_X___k1k2.nsimplify()
X_i_X_i_k1k2_X___k1k2 = factor(X_i_X_i_k1k2_X___k1k2)

X_i_X_i_k1k2_X_i_k1k2_in = T_3(tau_2, tau_3)*s_3(tau_2, tau_3)*(Q_11(tau_3)+2*Q_11(tau_2+tau_3))
X_i_X_i_k1k2_X_i_k1k2 = integrate(X_i_X_i_k1k2_X_i_k1k2_in, (tau_2, 0, oo), (tau_3, 0, oo))/(3*n)
X_i_X_i_k1k2_X_i_k1k2 = X_i_X_i_k1k2_X_i_k1k2.nsimplify()
X_i_X_i_k1k2_X_i_k1k2 = factor(X_i_X_i_k1k2_X_i_k1k2)

X_i_X_i_k1k2_X_j_k1k2 = (1/(n-1))*(X_i_X_i_k1k2_X___k1k2 - X_i_X_i_k1k2_X_i_k1k2)
X_i_X_i_k1k2_X_j_k1k2 = X_i_X_i_k1k2_X_j_k1k2.nsimplify()
X_i_X_i_k1k2_X_j_k1k2 = factor(X_i_X_i_k1k2_X_j_k1k2)

X_i_X_j_k1k2_X_r_k1k2 = (1/(n-2))*(X_i_k1k2_X_j_k1k2 - 2*X_i_X_i_k1k2_X_j_k1k2)
X_i_X_j_k1k2_X_r_k1k2 = X_i_X_j_k1k2_X_r_k1k2.nsimplify()
X_i_X_j_k1k2_X_r_k1k2 = factor(X_i_X_j_k1k2_X_r_k1k2)

X_i_X_j_k1k2_X_j_k1k2 = (1/(n-1))*(X_i_k1k2_X_i_k1k2 - X_i_X_i_k1k2_X_i_k1k2)
X_i_X_j_k1k2_X_j_k1k2 = X_i_X_j_k1k2_X_j_k1k2.nsimplify()
X_i_X_j_k1k2_X_j_k1k2 = factor(X_i_X_j_k1k2_X_j_k1k2)

In [31]:
X_i_k1k2_X___k1l2 = integrate(T_2(tau_2)*Q_10(tau_2), (tau_2, 0, oo))/n
X_i_k1k2_X___k1l2 = X_i_k1k2_X___k1l2.nsimplify()
X_i_k1k2_X___k1l2 = factor(X_i_k1k2_X___k1l2)

X_i_k1k2_X_i_k1l2 = integrate(T_2(tau_2)*s_2(tau_2)*Q_10(tau_2), (tau_2, 0, oo))/n
X_i_k1k2_X_i_k1l2 = X_i_k1k2_X_i_k1l2.nsimplify()
X_i_k1k2_X_i_k1l2 = factor(X_i_k1k2_X_i_k1l2)

X_i_k1k2_X_j_k1l2 = (1/(n-1))*(X_i_k1k2_X___k1l2 - X_i_k1k2_X_i_k1l2)
X_i_k1k2_X_j_k1l2 = X_i_k1k2_X_j_k1l2.nsimplify()
X_i_k1k2_X_j_k1l2 = factor(X_i_k1k2_X_j_k1l2)

In [32]:
X_i_X_i_k1k2_X___k1l2_in = T_3(tau_2, tau_3)*(s_2(tau_3)*Q_10(tau_2+tau_3)+s_2(tau_2+tau_3)*Q_10(tau_3)+s_2(tau_2+tau_3)*Q_10(tau_2+tau_3))
X_i_X_i_k1k2_X___k1l2 = integrate(X_i_X_i_k1k2_X___k1l2_in, (tau_2, 0, oo), (tau_3, 0, oo))/(3*n)
X_i_X_i_k1k2_X___k1l2 = X_i_X_i_k1k2_X___k1l2.nsimplify()
X_i_X_i_k1k2_X___k1l2 = factor(X_i_X_i_k1k2_X___k1l2)

X_i_X_i_k1k2_X_i_k1l2_in = T_3(tau_2, tau_3)*s_3(tau_2, tau_3)*(Q_10(tau_3)+2*Q_10(tau_2+tau_3))
X_i_X_i_k1k2_X_i_k1l2 = integrate(X_i_X_i_k1k2_X_i_k1l2_in, (tau_2, 0, oo), (tau_3, 0, oo))/(3*n)
X_i_X_i_k1k2_X_i_k1l2 = X_i_X_i_k1k2_X_i_k1l2.nsimplify()
X_i_X_i_k1k2_X_i_k1l2 = factor(X_i_X_i_k1k2_X_i_k1l2)

X_i_X_i_k1k2_X_j_k1l2 = (1/(n-1))*(X_i_X_i_k1k2_X___k1l2 - X_i_X_i_k1k2_X_i_k1l2)
X_i_X_i_k1k2_X_j_k1l2 = X_i_X_i_k1k2_X_j_k1l2.nsimplify()
X_i_X_i_k1k2_X_j_k1l2 = factor(X_i_X_i_k1k2_X_j_k1l2)

X_i_X_j_k1k2_X_r_k1l2 = (1/(n-2))*(X_i_k1k2_X_j_k1l2 - 2*X_i_X_i_k1k2_X_j_k1l2)
X_i_X_j_k1k2_X_r_k1l2 = X_i_X_j_k1k2_X_r_k1l2.nsimplify()
X_i_X_j_k1k2_X_r_k1l2 = factor(X_i_X_j_k1k2_X_r_k1l2)

X_i_X_j_k1k2_X_j_k1l2 = (1/(n-1))*(X_i_k1k2_X_i_k1l2 - X_i_X_i_k1k2_X_i_k1l2)
X_i_X_j_k1k2_X_j_k1l2 = X_i_X_j_k1k2_X_j_k1l2.nsimplify()
X_i_X_j_k1k2_X_j_k1l2 = factor(X_i_X_j_k1k2_X_j_k1l2)

In [33]:
X_i_k1k2_X___l1k2 = integrate(T_2(tau_2)*Q_01(tau_2), (tau_2, 0, oo))/n
X_i_k1k2_X___l1k2 = X_i_k1k2_X___l1k2.nsimplify()
X_i_k1k2_X___l1k2 = factor(X_i_k1k2_X___l1k2)

X_i_k1k2_X_i_l1k2 = integrate(T_2(tau_2)*s_2(tau_2)*Q_01(tau_2), (tau_2, 0, oo))/n
X_i_k1k2_X_i_l1k2 = X_i_k1k2_X_i_l1k2.nsimplify()
X_i_k1k2_X_i_l1k2 = factor(X_i_k1k2_X_i_l1k2)

X_i_k1k2_X_j_l1k2 = (1/(n-1))*(X_i_k1k2_X___l1k2 - X_i_k1k2_X_i_l1k2)
X_i_k1k2_X_j_l1k2 = X_i_k1k2_X_j_l1k2.nsimplify()
X_i_k1k2_X_j_l1k2 = factor(X_i_k1k2_X_j_l1k2)

In [34]:
X_i_X_i_k1k2_X___l1k2_in = T_3(tau_2, tau_3)*(s_2(tau_3)*Q_01(tau_2+tau_3)+s_2(tau_2+tau_3)*Q_01(tau_3)+s_2(tau_2+tau_3)*Q_01(tau_2+tau_3))
X_i_X_i_k1k2_X___l1k2 = integrate(X_i_X_i_k1k2_X___l1k2_in, (tau_2, 0, oo), (tau_3, 0, oo))/(3*n)
X_i_X_i_k1k2_X___l1k2 = X_i_X_i_k1k2_X___l1k2.nsimplify()
X_i_X_i_k1k2_X___l1k2 = factor(X_i_X_i_k1k2_X___l1k2)

X_i_X_i_k1k2_X_i_l1k2_in = T_3(tau_2, tau_3)*s_3(tau_2, tau_3)*(Q_01(tau_3)+2*Q_01(tau_2+tau_3))
X_i_X_i_k1k2_X_i_l1k2 = integrate(X_i_X_i_k1k2_X_i_l1k2_in, (tau_2, 0, oo), (tau_3, 0, oo))/(3*n)
X_i_X_i_k1k2_X_i_l1k2 = X_i_X_i_k1k2_X_i_l1k2.nsimplify()
X_i_X_i_k1k2_X_i_l1k2 = factor(X_i_X_i_k1k2_X_i_l1k2)

X_i_X_i_k1k2_X_j_l1k2 = (1/(n-1))*(X_i_X_i_k1k2_X___l1k2 - X_i_X_i_k1k2_X_i_l1k2)
X_i_X_i_k1k2_X_j_l1k2 = X_i_X_i_k1k2_X_j_l1k2.nsimplify()
X_i_X_i_k1k2_X_j_l1k2 = factor(X_i_X_i_k1k2_X_j_l1k2)

X_i_X_j_k1k2_X_r_l1k2 = (1/(n-2))*(X_i_k1k2_X_j_l1k2 - 2*X_i_X_i_k1k2_X_j_l1k2)
X_i_X_j_k1k2_X_r_l1k2 = X_i_X_j_k1k2_X_r_l1k2.nsimplify()
X_i_X_j_k1k2_X_r_l1k2 = factor(X_i_X_j_k1k2_X_r_l1k2)

X_i_X_j_k1k2_X_j_l1k2 = (1/(n-1))*(X_i_k1k2_X_i_l1k2 - X_i_X_i_k1k2_X_i_l1k2)
X_i_X_j_k1k2_X_j_l1k2 = X_i_X_j_k1k2_X_j_l1k2.nsimplify()
X_i_X_j_k1k2_X_j_l1k2 = factor(X_i_X_j_k1k2_X_j_l1k2)

## Calculation for coefficients $\lambda_i^m$ --- Eq.(S15)

In [36]:
lambda_1_1 = n*N**2*(X_i_X_j_k1k2_X_j_k1k2 + X_i_X_j_k1k2_X_j_k1l2 - X_i_X_j_k1k2_X_r_k1k2 - X_i_X_j_k1k2_X_r_k1l2)
lambda_1_1 = lambda_1_1.nsimplify()
lambda_1_1 = factor(lambda_1_1)

lambda_1_2 = n*N**2*(X_i_X_j_k1k2_X_j_k1k2 + X_i_X_j_k1k2_X_j_l1k2 - X_i_X_j_k1k2_X_r_k1k2 - X_i_X_j_k1k2_X_r_l1k2)
lambda_1_2 = lambda_1_2.nsimplify()
lambda_1_2 = factor(lambda_1_2)

lambda_2_1 = n*N**2*(X_i_X_i_k1k2_X_j_k1k2 + X_i_X_i_k1k2_X_j_k1l2 - X_i_X_j_k1k2_X_r_k1k2 - X_i_X_j_k1k2_X_r_k1l2)
lambda_2_1 = lambda_2_1.nsimplify()
lambda_2_1 = factor(lambda_2_1)

lambda_2_2 = n*N**2*(X_i_X_i_k1k2_X_j_k1k2 + X_i_X_i_k1k2_X_j_l1k2 - X_i_X_j_k1k2_X_r_k1k2 - X_i_X_j_k1k2_X_r_l1k2)
lambda_2_2 = lambda_2_2.nsimplify()
lambda_2_2 = factor(lambda_2_2)

lambda_3_1 = n**2 * N**2 * (X_i_X_j_k1k2_X_r_k1k2 + X_i_X_j_k1k2_X_r_k1l2)
lambda_3_1 = lambda_3_1.nsimplify()
lambda_3_1 = factor(lambda_3_1)

lambda_3_2 = n**2 * N**2 * (X_i_X_j_k1k2_X_r_k1k2 + X_i_X_j_k1k2_X_r_l1k2)
lambda_3_2 = lambda_3_2.nsimplify()
lambda_3_2 = factor(lambda_3_2)

## Calculation for $\sigma_m$ --- Eq.(S47)

### $\sigma_1$

In [39]:
def pi_1(p_1, p_2):
    return R_1*p_1*p_2 + S_1*p_1*(1-p_2) + T_1*(1-p_1)*p_2 + P_1*(1-p_1)*(1-p_2)

MM1 = factor(pi_1(p, p) - integrate(pi_1(q, q), (q, 0, 1)))
MM2 = factor(integrate(pi_1(p, q), (q, 0, 1)) - integrate(pi_1(q, p), (q, 0, 1)))
MM3 = factor(integrate(pi_1(p, q), (q, 0, 1)) - integrate(pi_1(p, q), (p, 0, 1), (q, 0, 1)))

MM1 = collect(MM1, (R_1, S_1, T_1, P_1))
MM2 = collect(MM2, (R_1, S_1, T_1, P_1))
MM3 = collect(MM3, (R_1, S_1, T_1, P_1))

D_p = factor(MM1*lambda_1_1 + MM2*lambda_2_1 + MM3*lambda_3_1)
p_Dp = factor(integrate(p*D_p, (p, 0, 1)))

propto_1 = (N**2*mu)/(24*n*r_1*(mu+1)*(nu+1)*(mu+nu+1)*(mu+nu+3))
p_Dp_propto = p_Dp/propto_1

# Calculate the coefficients of R_1, S_1, T_1, P_1
p_Dp_propto_R_1 = factor(p_Dp_propto.subs(S_1, 0).subs(T_1, 0).subs(P_1, 0).subs(R_1, 1))
p_Dp_propto_S_1 = factor(p_Dp_propto.subs(R_1, 0).subs(T_1, 0).subs(P_1, 0).subs(S_1, 1))
p_Dp_propto_T_1 = factor(p_Dp_propto.subs(S_1, 0).subs(R_1, 0).subs(P_1, 0).subs(T_1, 1))
p_Dp_propto_P_1 = factor(p_Dp_propto.subs(S_1, 0).subs(T_1, 0).subs(R_1, 0).subs(P_1, 1))

p_Dp_propto_R_1 = collect(p_Dp_propto_R_1, r_1)
p_Dp_propto_S_1 = collect(p_Dp_propto_S_1, r_1)
p_Dp_propto_T_1 = collect(p_Dp_propto_T_1, r_1)
p_Dp_propto_P_1 = collect(p_Dp_propto_P_1, r_1)

In [40]:
p_Dp_propto_R_1

(mu + nu + 1)*(mu*nu + nu**2 + 2*nu + r_1*(mu + 2*nu + 3))

In [41]:
p_Dp_propto_S_1

(mu + nu + 3)*(mu*nu + nu**2 + 2*nu + r_1*(mu + 1))

In [42]:
p_Dp_propto_T_1

-(mu + nu + 3)*(mu*nu + nu**2 + 2*nu + r_1*(mu + 1))

In [43]:
p_Dp_propto_P_1

-(mu + nu + 1)*(mu*nu + nu**2 + 2*nu + r_1*(mu + 2*nu + 3))

### $\sigma_2$

In [46]:
def pi_2(q_1, q_2):
    return R_2*q_1*q_2 + S_2*q_1*(1-q_2) + TT_2*(1-q_1)*q_2 + P_2*(1-q_1)*(1-q_2)

NN1 = factor(pi_2(q, q) - integrate(pi_2(p, p), (p, 0, 1)))
NN2 = factor(integrate(pi_2(q, p), (p, 0, 1)) - integrate(pi_2(p, q), (p, 0, 1)))
NN3 = factor(integrate(pi_2(q, p), (p, 0, 1)) - integrate(pi_2(p, q), (p, 0, 1), (q, 0, 1)))

NN1 = collect(NN1, (R_2, S_2, TT_2, P_2))
NN2 = collect(NN2, (R_2, S_2, TT_2, P_2))
NN3 = collect(NN3, (R_2, S_2, TT_2, P_2))

D_q = factor(NN1*lambda_1_2 + NN2*lambda_2_2 + NN3*lambda_3_2)
q_Dq = factor(integrate(q*D_q, (q, 0, 1)))

propto_2 = (N**2*mu)/(24*n*R_K*(mu+1)*(nu+1)*(mu+nu+1)*(mu+nu+3))
q_Dq_propto = q_Dq/propto_2

# Calculate the coefficients of R_2, S_2, T_2, P_2
q_Dq_propto_R_2 = factor(q_Dq_propto.subs(S_2, 0).subs(TT_2, 0).subs(P_2, 0).subs(R_2, 1))
q_Dq_propto_S_2 = factor(q_Dq_propto.subs(R_2, 0).subs(TT_2, 0).subs(P_2, 0).subs(S_2, 1))
q_Dq_propto_T_2 = factor(q_Dq_propto.subs(S_2, 0).subs(R_2, 0).subs(P_2, 0).subs(TT_2, 1))
q_Dq_propto_P_2 = factor(q_Dq_propto.subs(S_2, 0).subs(TT_2, 0).subs(R_2, 0).subs(P_2, 1))

q_Dq_propto_R_2 = collect(q_Dq_propto_R_2, R_K)
q_Dq_propto_S_2 = collect(q_Dq_propto_S_2, R_K)
q_Dq_propto_T_2 = collect(q_Dq_propto_T_2, R_K)
q_Dq_propto_P_2 = collect(q_Dq_propto_P_2, R_K)

In [47]:
q_Dq_propto_R_2.subs(R_K, 2*K+1)

(mu + nu + 1)*(mu*nu + nu**2 + 2*nu + (2*K + 1)*(mu + 2*nu + 3))

In [48]:
q_Dq_propto_S_2.subs(R_K, 2*K+1)

(mu + nu + 3)*(mu*nu + nu**2 + 2*nu + (2*K + 1)*(mu + 1))

In [49]:
q_Dq_propto_T_2.subs(R_K, 2*K+1)

-(mu + nu + 3)*(mu*nu + nu**2 + 2*nu + (2*K + 1)*(mu + 1))

In [50]:
q_Dq_propto_P_2.subs(R_K, 2*K+1)

-(mu + nu + 1)*(mu*nu + nu**2 + 2*nu + (2*K + 1)*(mu + 2*nu + 3))